# Estruturação da Bíblia em versículos para o vectorstore

Este notebook prepara a Bíblia Ave-Maria para uso em busca semântica e recuperação de contexto. Ele extrai o texto do PDF, remove ruídos de formatação, organiza o conteúdo em livro, capítulo e versículo, e gera uma base mais confiável para indexação.

## Objetivos

- Extrair o texto bruto do PDF ou reutilizar o TXT já processado.
- Remover cabeçalhos, rodapés e trechos indesejados.
- Identificar livros, capítulos e versículos com consistência.
- Produzir uma saída estruturada para o pipeline de vectorstore.


In [1]:
BIBLIA_INPUT_PDF = (
    "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.pdf"
)
BIBLIA_TXT_EXTRACTED = (
    "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
)
BIBLIA_TEMP = (
    "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-temp.txt"
)
BIBLIA_OUTPUT = (
    "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
)

In [2]:
PROCESSAR_PDF = False
PROCESSAR_TXT = True

In [3]:
import pdfplumber


def extrair_texto_pdf(pdf_path, txt_path):
    with pdfplumber.open(pdf_path) as pdf:
        with open(txt_path, "w", encoding="utf-8") as f_out:
            for i, page in enumerate(pdf.pages):
                if i < 2:
                    continue
                elif i == 2904:
                    break
                texto = page.extract_text()
                if texto:
                    f_out.write(texto + "\n")
    print(f"Texto extraído para: {txt_path}")


if PROCESSAR_PDF:
    extrair_texto_pdf(BIBLIA_INPUT_PDF, BIBLIA_TXT_EXTRACTED)

In [4]:
def remover_3_primeiras_linhas(caminho_entrada, caminho_saida):
    with open(caminho_entrada, "r", encoding="utf-8") as f_in:
        linhas = f_in.readlines()
    with open(caminho_saida, "w", encoding="utf-8") as f_out:
        f_out.writelines(linhas[3:])
    print(f"Texto processado para: {caminho_entrada}")


if PROCESSAR_TXT:
    remover_3_primeiras_linhas(BIBLIA_TXT_EXTRACTED, BIBLIA_TEMP)

Texto processado para: ../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt


In [5]:
# def remover_trechos_indesejados(caminho):
#     trecho_alvo_1 = "Bíblia Ave-Maria"
#     trecho_alvo_2 = "© 1959 Editora Ave-Maria."
#     remover_linhas = False

#     with open(caminho, "r", encoding="utf-8") as f:
#         linhas = f.readlines()

#     linhas_filtradas = []
#     for linha in linhas:
#         if trecho_alvo_1 in linha:
#             remover_linhas = True
#         elif remover_linhas and trecho_alvo_2 in linha:
#             continue
#         elif remover_linhas and linha.strip().isdigit():
#             remover_linhas = False
#             continue
#         if not remover_linhas:
#             linhas_filtradas.append(linha)

#     with open(caminho, "w", encoding="utf-8") as f:
#         f.writelines(linhas_filtradas)

#     print(f"Trechos indesejados removidos de {caminho}")


# if PROCESSAR_TXT:
#     remover_trechos_indesejados(BIBLIA_TEMP)

In [6]:
map_proximo_livro = {
    "Livro Inicial": "Gênesis",
    # Antigo Testamento
    "Gênesis": "Êxodo",
    "Êxodo": "Levítico",
    "Levítico": "Números",
    "Números": "Deuteronômio",
    "Deuteronômio": "Josué",
    "Josué": "Juízes",
    "Juízes": "Rute",
    "Rute": "1 Samuel",
    "1 Samuel": "2 Samuel",
    "2 Samuel": "1 Reis",
    "1 Reis": "2 Reis",
    "2 Reis": "1 Crônicas",
    "1 Crônicas": "2 Crônicas",
    "2 Crônicas": "Esdras",
    "Esdras": "Neemias",
    "Neemias": "Tobias",
    "Tobias": "Judite",
    "Judite": "Ester",
    "Ester": "Jó",
    "Jó": "Salmos",
    "Salmos": "1 Macabeus",
    "1 Macabeus": "2 Macabeus",
    "2 Macabeus": "Provérbios",
    "Provérbios": "Eclesiastes",
    "Eclesiastes": "Cântico dos Cânticos",
    "Cântico dos Cânticos": "Sabedoria",
    "Sabedoria": "Eclesiástico",
    "Eclesiástico": "Isaías",
    "Isaías": "Jeremias",
    "Jeremias": "Lamentações de Jeremias",
    "Lamentações de Jeremias": "Baruc",
    "Baruc": "Ezequiel",
    "Ezequiel": "Daniel",
    "Daniel": "Oseias",
    "Oseias": "Joel",
    "Joel": "Amós",
    "Amós": "Abdias",
    "Abdias": "Jonas",
    "Jonas": "Miqueias",
    "Miqueias": "Naum",
    "Naum": "Habacuc",
    "Habacuc": "Sofonias",
    "Sofonias": "Ageu",
    "Ageu": "Zacarias",
    "Zacarias": "Malaquias",
    "Malaquias": "São Mateus",
    # Novo Testamento
    "São Mateus": "São Marcos",
    "São Marcos": "São Lucas",
    "São Lucas": "São João",
    "São João": "Atos",
    "Atos": "Romanos",
    "Romanos": "1 Coríntios",
    "1 Coríntios": "2 Coríntios",
    "2 Coríntios": "Gálatas",
    "Gálatas": "Efésios",
    "Efésios": "Filipenses",
    "Filipenses": "Colossenses",
    "Colossenses": "1 Tessalonicenses",
    "1 Tessalonicenses": "2 Tessalonicenses",
    "2 Tessalonicenses": "1 Timóteo",
    "1 Timóteo": "2 Timóteo",
    "2 Timóteo": "Tito",
    "Tito": "Filemon",
    "Filemon": "Hebreus",
    "Hebreus": "Tiago",
    "Tiago": "1 São Pedro",
    "1 São Pedro": "2 São Pedro",
    "2 São Pedro": "1 São João",
    "1 São João": "2 São João",
    "2 São João": "3 São João",
    "3 São João": "São Judas",
    "São Judas": "Apocalipse",
}

In [7]:
import pandas as pd


def adicionar_versiculo(
    df: pd.DataFrame,
    book: str,
    chapter: int,
    verse: int,
    text: str,
    verse_acc: int,
    pdf_page: int,
    need_review: bool,
) -> pd.DataFrame:
    """Adiciona versículo ao dataframe"""    
    new_row = pd.DataFrame(
        [
            {
                "book": book,
                "chapter": chapter,
                "verse": verse,
                "text": text,
                "verse_acc": verse_acc,
                "pdf_page": pdf_page,
                "need_review": need_review,
            }
        ]
    )
    
    # print("Adicionando versículo: {book} {chapter}:{verse} - {text}".format(
    #     book=book, chapter=chapter, verse=verse, text=text
    # ))
    
    return pd.concat([df, new_row], ignore_index=True)


def escrever_versiculo_no_arquivo():
    """Escreve versículo em arquivo.

    Formato do versículo: nome_livro numero_capitulo:nume_versiculo texto_versiculo
    """
    pass

# Estruturação confiável de versículos bíblicos

Para elaborar um parser melhor, eu seguiria estas instruções:

- [ ] Defina o modelo de saída antes de extrair: cada registro deve ter livro canônico, capítulo, versículo, texto, página do PDF, linha bruta e um indicador de confiança.
- [ ] Extraia preservando contexto de página. Cada fragmento deve carregar o número da página e, idealmente, posição/ordem no PDF, para que qualquer erro seja rastreável até a imagem original.
- [ ] Normalize a fonte com cautela: remova cabeçalho, rodapé e número de página apenas quando coincidirem com padrões confirmados e posição de página; não descarte toda linha numérica genericamente.
- [ ] Reconheça explicitamente os tipos de marcador: capítulo (`nome de livro + capítulo`), versículo simples, intervalo (`2-7`), faixa compacta ou agrupada, títulos, notas e referências cruzadas. Qualquer linha que não se enquadre deve virar texto ou ir para revisão, nunca marcador por suposição.
- [ ] Mantenha intervalos como intervalos. Para `Josué 19:2-7`, prefira guardar um único registro com início 2 e fim 7, ou dividir em `vv. 2–7` somente se o PDF trouxer separação textual confiável. Não invente a distribuição.
- [ ] Valide a sequência em cada capítulo. Registre alertas para repetição, salto, retrocesso, intervalo sobreposto e número improvável; a validação deve apontar `8 → 10 → 10`, não corrigir automaticamente.
- [ ] Tenha regras explícitas para exceções bíblicas: Salmos, livros de um capítulo, títulos de salmos, versículos agrupados, notas e variações de nomenclatura dos livros.
- [ ] Preserve o original e a versão limpa. Nunca substitua o texto bruto: mantenha uma camada de extração, uma de segmentação e outra de normalização, para que uma correção não exija reextrair tudo.
- [ ] Crie uma fila de revisão humana para casos ambíguos. Em vez de decidir silenciosamente, marque registros com baixa confiança, como `2-7` convertido em `27` ou repetições de número.
- [ ] Compare a estrutura final com uma referência canônica de contagem por livro/capítulo. A meta não é substituir o texto da Ave-Maria, mas verificar se as chaves de referência estão completas e coerentes.
- [ ] Faça testes de regressão com páginas problemáticas: `Josué 19`, `2 Reis 3`, páginas com rodapé, quebras por hifenização, início/fim de livro e páginas com títulos. Cada erro corrigido deve virar um caso permanente de teste.
- [ ] Mantenha a base canônica em registros atômicos por versículo, com ID estável (`tradução`, livro, capítulo, versículo), e represente `2-7` como uma referência/ocorrência que aponta para os IDs `2`, `3`, `4`, `5`, `6` e `7`.
- [ ] Se o texto fonte não permitir separar com segurança cada versículo, marque o trecho como `NEEDS_REVIEW`, preservando o intervalo e a fonte bruta, em vez de inventar a divisão.
- [ ] Use uma fonte católica, licenciada e versionada, não apenas "canônica".
- [ ] Valide comparando SQLite e Chroma, garantindo que o índice seja recriado somente a partir da base certificada.

## Critérios de conclusão

- [ ] A extração preserva o original, a versão limpa, a rastreabilidade por página e a fila de revisão humana.
- [ ] A base canônica permanece atômica por versículo, com ocorrências de intervalos mapeadas sem inventar divisão textual.
- [ ] SQLite e Chroma batem exatamente contra a base certificada antes de qualquer publicação ou reindexação.


In [8]:
import re
import os

from bible_model import Verse


def transformar_biblia_versiculo_a_versiculo(caminho_entrada, caminho_saida):
    livro_atual = "Livro Inicial"
    capitulo_atual = 0
    versiculo_atual = 0
    ultimo_versiculo = 0
    versiculo_acc = 0
    mark_new_page = False
    pagina_atual = 3

    buffer_versiculo = []

    df_verse = pd.DataFrame(
        {
            "book": pd.Series(dtype="Int64"),
            "chapter": pd.Series(dtype="Int64"),
            "verse": pd.Series(dtype="Int64"),
            "text": pd.Series(dtype="string"),
            "verse_acc": pd.Series(dtype="Int64"),
            "pdf_page": pd.Series(dtype="Int64"),
            "need_review": pd.Series(dtype="boolean"),
        }
    )

    # Regex para detectar linhas como "Gênesis 1"
    # Esse regex é capaz de detectar trechos como:
    # - "Gênesis 1"
    # - "1 Samuel 2"
    # - "Cântico dos Cânticos 3"
    regex_livro_capitulo = re.compile(r"^(.*)\s+(\d+)$")

    def salvar_buffer_versiculo():
        nonlocal df_verse, buffer_versiculo, versiculo_atual, ultimo_versiculo

        if versiculo_atual == 0 or not buffer_versiculo:
            buffer_versiculo = []
            return

        # Alguns versículos ficam quebrados em várias linhas no texto bruto.
        # O buffer só é persistido quando o próximo marcador aparece.
        if versiculo_atual == 1 or versiculo_atual == ultimo_versiculo + 1:
            need_review = False
        else:
            need_review = True

        df_verse = adicionar_versiculo(
            df_verse,
            livro_atual,
            capitulo_atual,
            versiculo_atual,
            "\n".join(buffer_versiculo),
            versiculo_acc,
            pagina_atual,
            need_review,
        )
        ultimo_versiculo = versiculo_atual
        buffer_versiculo = []

    def iniciar_novo_versiculo(numero_versiculo):
        nonlocal versiculo_atual, versiculo_acc, buffer_versiculo

        salvar_buffer_versiculo()
        versiculo_atual = numero_versiculo
        versiculo_acc += 1
        buffer_versiculo = []

    with open(caminho_entrada, "r", encoding="utf-8") as entrada:
        for linha in entrada:
            linha = linha.strip()

            if not linha:
                continue  # pula linhas vazias

            # print(f"Linha atual: {linha}")

            # Verifica nova página (PDF)
            if linha == "Bíblia Ave-Maria" or linha == "© 1959 Editora Ave-Maria.":
                # A troca de página pode acontecer no meio de um versículo.
                # Nesse caso, mantemos o buffer aberto para unificar o texto.
                mark_new_page = True
                continue
            if mark_new_page and linha.isdigit():
                pagina_atual = int(linha)
                # print(f"Nova página detectada: {pagina_atual}")
                mark_new_page = False
                continue

            # Verifica se é início de livro (só nome do livro)
            if (
                livro_atual
                != "Apocalipse"  # Apocalipse é o último livro, não há próximo mapeado!
                and linha.startswith(map_proximo_livro[livro_atual])
                and len(linha.split()) <= 3
            ):
                salvar_buffer_versiculo()
                print(f"Livro encontrado: {linha}")
                livro_atual = linha.strip()
                capitulo_atual = 0
                versiculo_atual = 0
                ultimo_versiculo = 0
                buffer_versiculo = []
                continue

            # Verifica se é início de capítulo ("Livro Capítulo")
            match = regex_livro_capitulo.match(linha)
            if match:
                # print(f"Capítulo encontrado: {match.group(1)} {match.group(2)}")
                salvar_buffer_versiculo()
                capitulo_atual = int(match.group(2).strip())
                versiculo_atual = 0
                ultimo_versiculo = 0
                buffer_versiculo = []
                continue

            # Verifica se é um número (versículo)
            elif linha.strip().isdigit():
                # print(f"Número encontrado: {linha}"
                iniciar_novo_versiculo(int(linha.strip()))
                continue
            
            # Daqui pra frente estamos lidando com 1 ou mais linhas que formam um único versículo.
            # As quebras de linha internas são preservadas até surgir o próximo marcador.
            if versiculo_atual == 0:
                continue

            buffer_versiculo.append(linha)

    # Salva o último versículo
    salvar_buffer_versiculo()

    print(f"Transformação concluída. Versículos salvos em {caminho_saida}")
    return df_verse


In [9]:
if PROCESSAR_TXT:
    df_verse = transformar_biblia_versiculo_a_versiculo(
        BIBLIA_TEMP,
        BIBLIA_OUTPUT,
    )

    if PROCESSAR_TXT and os.path.exists(BIBLIA_TEMP):
        os.remove(BIBLIA_TEMP)
        print(f"Arquivo temporário removido: {BIBLIA_TEMP}")

Livro encontrado: Gênesis
Livro encontrado: Êxodo
Livro encontrado: Levítico
Livro encontrado: Números
Livro encontrado: Deuteronômio
Livro encontrado: Josué
Livro encontrado: Juízes
Livro encontrado: Rute
Livro encontrado: 1 Samuel
Livro encontrado: 2 Samuel
Livro encontrado: 1 Reis
Livro encontrado: 2 Reis
Livro encontrado: 1 Crônicas
Livro encontrado: 2 Crônicas
Livro encontrado: Esdras
Livro encontrado: Neemias
Livro encontrado: Tobias
Livro encontrado: Judite
Livro encontrado: Ester
Livro encontrado: Jó
Livro encontrado: Salmos
Livro encontrado: 1 Macabeus
Livro encontrado: 2 Macabeus
Livro encontrado: Provérbios
Livro encontrado: Eclesiastes
Livro encontrado: Cântico dos Cânticos
Livro encontrado: Sabedoria
Livro encontrado: Eclesiástico
Livro encontrado: Isaías
Livro encontrado: Jeremias
Livro encontrado: Lamentações de Jeremias
Livro encontrado: Baruc
Livro encontrado: Ezequiel
Livro encontrado: Daniel
Livro encontrado: Oseias
Livro encontrado: Joel
Livro encontrado: Amós
Livro

## Análise e Pós-processamento

In [10]:
if not PROCESSAR_TXT:
    raise ValueError(
        "PROCESSAR_TXT Falso, PARE O NOTEBOOK AQUI!\\Não prosseguir com análise e pós-processamento."
    )

In [11]:
df_verse

,book,chapter,verse,text,verse_acc,pdf_page,need_review
0,Gênesis,1,1,"No princípio, Deus criou o céu e a terra.",1,3,False
1,Gênesis,1,2,A terra estava sem forma e vazia; as trevas co...,2,3,False
2,Gênesis,1,3,Deus disse: “Faça-se a luz!”. E a luz foi feita.,3,3,False
3,Gênesis,1,4,"Deus viu que a luz era boa, e separou a luz da...",4,3,False
4,Gênesis,1,5,"Deus chamou à luz dia, e às trevas noite. Sobr...",5,3,False
...,...,...,...,...,...,...,...
35492,Apocalipse,22,17,O Espírito e a Esposa dizem: “Vem!”. Possa aqu...,35493,2904,False
35493,Apocalipse,22,18,Eu declaro a todos aqueles que ouvirem as pala...,35494,2904,False
35494,Apocalipse,22,19,"e, se alguém dele tirar qualquer coisa, Deus l...",35495,2904,False
35495,Apocalipse,22,20,Aquele que atesta estas coisas diz: “Sim! Eu v...,35496,2904,False


In [12]:
df_verse.head(10)

,book,chapter,verse,text,verse_acc,pdf_page,need_review
0,Gênesis,1,1,"No princípio, Deus criou o céu e a terra.",1,3,False
1,Gênesis,1,2,A terra estava sem forma e vazia; as trevas co...,2,3,False
2,Gênesis,1,3,Deus disse: “Faça-se a luz!”. E a luz foi feita.,3,3,False
3,Gênesis,1,4,"Deus viu que a luz era boa, e separou a luz da...",4,3,False
4,Gênesis,1,5,"Deus chamou à luz dia, e às trevas noite. Sobr...",5,3,False
5,Gênesis,1,6,Deus disse: “Faça-se um firmamento entre as ág...,6,3,False
6,Gênesis,1,7,Deus fez o firmamento e separou as águas que e...,7,3,False
7,Gênesis,1,8,E assim se fez. Deus chamou ao firmamento céu....,8,3,False
8,Gênesis,1,9,Deus disse: “Que as águas que estão debaixo do...,9,3,False
9,Gênesis,1,10,"Deus chamou ao elemento árido terra, e ao ajun...",10,3,False


In [13]:
df_verse.tail(10)

,book,chapter,verse,text,verse_acc,pdf_page,need_review
35487,Apocalipse,22,12,"Eis que venho em breve, e a minha recompensa e...",35488,2903,False
35488,Apocalipse,22,13,"Eu sou o Alfa e o Ômega, o Primeiro e o Último...",35489,2903,False
35489,Apocalipse,22,14,Felizes aqueles que lavam as suas vestes para ...,35490,2903,False
35490,Apocalipse,22,15,"Fora os cães, os envenenadores, os impudicos, ...",35491,2904,False
35491,Apocalipse,22,16,"Eu, Jesus, enviei o meu anjo para vos atestar ...",35492,2904,False
35492,Apocalipse,22,17,O Espírito e a Esposa dizem: “Vem!”. Possa aqu...,35493,2904,False
35493,Apocalipse,22,18,Eu declaro a todos aqueles que ouvirem as pala...,35494,2904,False
35494,Apocalipse,22,19,"e, se alguém dele tirar qualquer coisa, Deus l...",35495,2904,False
35495,Apocalipse,22,20,Aquele que atesta estas coisas diz: “Sim! Eu v...,35496,2904,False
35496,Apocalipse,22,21,A graça do Senhor Jesus esteja com todos.,35497,2904,False


In [14]:
df_verse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35497 entries, 0 to 35496
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   book         35497 non-null  object 
 1   chapter      35497 non-null  Int64  
 2   verse        35497 non-null  Int64  
 3   text         35497 non-null  object 
 4   verse_acc    35497 non-null  Int64  
 5   pdf_page     35497 non-null  Int64  
 6   need_review  35497 non-null  boolean
dtypes: Int64(4), boolean(1), object(2)
memory usage: 1.8+ MB


In [15]:
df_verse[df_verse["need_review"]]

,book,chapter,verse,text,verse_acc,pdf_page,need_review
6139,Josué,12,924,"- Foram eles: o rei de Jericó, o rei de Hai, p...",6140,499,True
6209,Josué,15,2232,"- Cabseel, Arad, Jagur, Cina, Dimona, Adada, C...",6210,505,True
6210,Josué,15,3347,"- Na planície: Estaol, Saraá, Asena, Zanoe, En...",6211,506,True
6211,Josué,15,4860,"- Na montanha: Samir, Jeter, Sucot, Dana, Cari...",6212,506,True
6212,Josué,15,61,"No deserto: Bet-Arabá, Medin Sacaca,",6213,506,True
6263,Josué,18,2128,"- As cidades da tribo dos benjaminitas, segund...",6264,511,True
6265,Josué,19,27,"- Tiveram por herança: Bersabeia (Sabeia), Mol...",6266,511,True
6266,Josué,19,8,assim como todos os lugarejos dos arredores de...,6267,511,True
6275,Josué,19,1723,"- A quarta sorte coube a Issacar, aos filhos d...",6276,512,True
6276,Josué,19,24,A quinta sorte coube à tribo dos filhos de Ase...,6277,512,True


In [16]:
# Traz todos os need_review, junto à linha anterior e à próxima linha, já deduplicando caso choque
need_review_pos = df_verse.index[df_verse["need_review"].fillna(False)]

context_pos = set()
for pos in need_review_pos:
    context_pos.update({pos - 1, pos, pos + 1})

context_pos = [pos for pos in sorted(context_pos) if 0 <= pos < len(df_verse)]
df_verse.iloc[context_pos]

,book,chapter,verse,text,verse_acc,pdf_page,need_review
6138,Josué,12,8,"tanto na montanha como nas planícies, e sobre ...",6139,499,False
6139,Josué,12,924,"- Foram eles: o rei de Jericó, o rei de Hai, p...",6140,499,True
6140,Josué,13,1,"Josué estava velho, avançado em anos, e o Senh...",6141,499,False
6208,Josué,15,21,As cidades situadas na extremidade da tribo do...,6209,505,False
6209,Josué,15,2232,"- Cabseel, Arad, Jagur, Cina, Dimona, Adada, C...",6210,505,True
...,...,...,...,...,...,...,...
22451,Isaías,41,8,"Mas tu, Israel, meu servo, Jacó que escolhi, r...",22452,1814,True
22452,Isaías,41,9,"tu, que eu trouxe dos confins da terra, e que ...",22453,1814,False
26916,Jonas,1,11,E disseram-lhe: “Que te havemos de fazer para ...,26917,2216,False
26917,Jonas,1,13,Os homens remavam para ver se conseguiam ganha...,26918,2216,True
